In [6]:
import re
import pandas as pd

DATA_DIR = '/home/jmbyeon/developments/tf_litigation/MANTF-master/src'

## 1. Cases — case_type_1 == 1 필터

In [7]:
cases = pd.read_csv(f'{DATA_DIR}/cases.csv')
cases_type1 = cases[cases['case_type_1'] == 1][['case_row_id', 'case_number']].copy()
print(f'case_type_1==1 케이스 수: {len(cases_type1):,}')
cases_type1.head()

case_type_1==1 케이스 수: 46,907


,case_row_id,case_number
2,727,0:03-cv-00968
3,728,0:03-cv-00993
4,729,0:03-cv-01016
6,731,0:03-cv-01064
7,732,0:03-cv-01160


## 2. Names — Plaintiff / Defendant 피벗

In [8]:
names = pd.read_csv(f'{DATA_DIR}/names.csv')

# contains 매칭: Counter Plaintiff, Consol Defendant 등 모두 포함
def party_side(pt):
    pt = str(pt).lower()
    if 'plaintiff' in pt: return 'Plaintiff'
    if 'defendant' in pt: return 'Defendant'
    return None

names['party_side'] = names['party_type'].apply(party_side)
names_filtered = names[names['party_side'].notna()][['case_row_id', 'party_side', 'name']]

print(f'정확 매칭(기존): {names["party_type"].isin(["Plaintiff","Defendant"]).sum():,}행')
print(f'contains 매칭(신규): {names_filtered.shape[0]:,}행  (+{names_filtered.shape[0] - names["party_type"].isin(["Plaintiff","Defendant"]).sum():,})')
print()

# 케이스마다 같은 side가 여러 행일 수 있으므로 이름 합치기
names_grouped = (
    names_filtered
    .groupby(['case_row_id', 'party_side'])['name']
    .apply(lambda x: ' | '.join(x.dropna().astype(str)))
    .reset_index()
)

# 피벗: party_side → 열로
names_pivot = names_grouped.pivot(index='case_row_id', columns='party_side', values='name').reset_index()
names_pivot.columns.name = None
names_pivot.rename(columns={'Plaintiff': 'plaintiff', 'Defendant': 'defendant'}, inplace=True)

print(f'names_pivot 행 수: {len(names_pivot):,}')
names_pivot.head()

정확 매칭(기존): 447,533행
contains 매칭(신규): 586,033행  (+138,500)

names_pivot 행 수: 79,408


,case_row_id,defendant,plaintiff
0,1,Generix Drug Corp.,Burroghs Wellcome Co.
1,3,"U. S. COnstruction Manufacturing, Inc.",Kenneth R. Cornwall
2,4,On-Rite Hairpiece Company | Andrew O. Wright,"Monte Carlo Hairpieces, Inc. | James L. Waters"
3,5,"Atlantic Federal S & L Assn., Inc. | Ozark Fin...","Monaco Del, Rocco A. Sr. | Frances Monaco Del"
4,6,Sid Malkin | Morton Malkin | True Fit Chair Of...,"Contour Chair Lounge,Co.,Inc. | Craftmatic/Con..."


## 3. Cases + Names 병합

In [9]:
cases_with_parties = cases_type1.merge(names_pivot, on='case_row_id', how='left')
print(f'병합 후 행 수: {len(cases_with_parties):,}')
cases_with_parties.head(10)

병합 후 행 수: 46,907


,case_row_id,case_number,defendant,plaintiff
0,727,0:03-cv-00968,Michael Degerness | Advanced Barrel Systems | ...,"Magnum Research, Inc"
1,728,0:03-cv-00993,Alumacraft Boat Co,"Crestliner, Inc. | Crestliner Inc"
2,729,0:03-cv-01016,"AS Hospitality, Inc.",Taylor Corporation
3,731,0:03-cv-01064,Koninklijke Philips Electronics N.V. | a Nethe...,"Cardiac Science, Inc | a Delaware Corporation ..."
4,732,0:03-cv-01160,"Anderson Environmental Systems, Inc. | Andex C...","Ecolab Inc. | Ecolab, Inc."
5,733,0:03-cv-01183,Wagner Spray Tech Corporation | Diversified Dy...,Diversified Dynamics Corporation
6,734,0:03-cv-01189,"Medical Research Laboratories, Inc. | an Illin...","Cardiac Science, Inc | a Delaware corporation ..."
7,736,0:03-cv-02231,"JohnsonDiversey | JohnsonDiversey, Inc. | Ecol...","Ecolab, Inc."
8,738,0:03-cv-02477,KV Pharmaceutical Company | ETHEX Corporation ...,CIMA Labs Inc | Schwarz Pharma Inc | Cima Labs...
9,739,0:03-cv-02499,"Big Roc Tools, Inc.",Bondhus Corporation


## 4. Patents — 특허 번호 연결

In [10]:
patents = pd.read_csv(f'{DATA_DIR}/patents.csv')

# 케이스마다 특허가 여러 개일 수 있으므로 합치기
patents_grouped = (
    patents[['case_row_id', 'patent']]
    .groupby('case_row_id')['patent']
    .apply(lambda x: ' | '.join(x.dropna().astype(str)))
    .reset_index()
    .rename(columns={'patent': 'patents'})
)

print(f'patents_grouped 행 수: {len(patents_grouped):,}')
patents_grouped.head()

patents_grouped 행 수: 54,705


,case_row_id,patents
0,725,5696571 | 6247718
1,726,6384088 | 6403657
2,727,6457274 | 5600912
3,728,5108321
4,729,6153278


## 5. 최종 데이터셋

In [11]:
result = cases_with_parties.merge(patents_grouped, on='case_row_id', how='left')
print(f'최종 행 수: {len(result):,}')
result.head(20)

최종 행 수: 46,907


,case_row_id,case_number,defendant,plaintiff,patents
0,727,0:03-cv-00968,Michael Degerness | Advanced Barrel Systems | ...,"Magnum Research, Inc",6457274 | 5600912
1,728,0:03-cv-00993,Alumacraft Boat Co,"Crestliner, Inc. | Crestliner Inc",5108321
2,729,0:03-cv-01016,"AS Hospitality, Inc.",Taylor Corporation,6153278
3,731,0:03-cv-01064,Koninklijke Philips Electronics N.V. | a Nethe...,"Cardiac Science, Inc | a Delaware Corporation ...",5897576 | 5800460 | 6016059 | 5591213 | 577396...
4,732,0:03-cv-01160,"Anderson Environmental Systems, Inc. | Andex C...","Ecolab Inc. | Ecolab, Inc.",5365690 | 5505017
5,733,0:03-cv-01183,Wagner Spray Tech Corporation | Diversified Dy...,Diversified Dynamics Corporation,4810123
6,734,0:03-cv-01189,"Medical Research Laboratories, Inc. | an Illin...","Cardiac Science, Inc | a Delaware corporation ...",5579919
7,736,0:03-cv-02231,"JohnsonDiversey | JohnsonDiversey, Inc. | Ecol...","Ecolab, Inc.",6495494 | 6743758 | 6673753
8,738,0:03-cv-02477,KV Pharmaceutical Company | ETHEX Corporation ...,CIMA Labs Inc | Schwarz Pharma Inc | Cima Labs...,6024981
9,739,0:03-cv-02499,"Big Roc Tools, Inc.",Bondhus Corporation,D382190 | 5970828 | 5791211


In [12]:
# 결측 현황 확인
result.isna().sum()

case_row_id      0
case_number      0
defendant      633
plaintiff      810
patents          0
dtype: int64

In [13]:
# 특허 번호 있는 케이스만
result_with_patent = result.dropna(subset=['patents'])
print(f'특허 번호 있는 케이스 수: {len(result_with_patent):,}')
result_with_patent.head(20)

특허 번호 있는 케이스 수: 46,907


,case_row_id,case_number,defendant,plaintiff,patents
0,727,0:03-cv-00968,Michael Degerness | Advanced Barrel Systems | ...,"Magnum Research, Inc",6457274 | 5600912
1,728,0:03-cv-00993,Alumacraft Boat Co,"Crestliner, Inc. | Crestliner Inc",5108321
2,729,0:03-cv-01016,"AS Hospitality, Inc.",Taylor Corporation,6153278
3,731,0:03-cv-01064,Koninklijke Philips Electronics N.V. | a Nethe...,"Cardiac Science, Inc | a Delaware Corporation ...",5897576 | 5800460 | 6016059 | 5591213 | 577396...
4,732,0:03-cv-01160,"Anderson Environmental Systems, Inc. | Andex C...","Ecolab Inc. | Ecolab, Inc.",5365690 | 5505017
5,733,0:03-cv-01183,Wagner Spray Tech Corporation | Diversified Dy...,Diversified Dynamics Corporation,4810123
6,734,0:03-cv-01189,"Medical Research Laboratories, Inc. | an Illin...","Cardiac Science, Inc | a Delaware corporation ...",5579919
7,736,0:03-cv-02231,"JohnsonDiversey | JohnsonDiversey, Inc. | Ecol...","Ecolab, Inc.",6495494 | 6743758 | 6673753
8,738,0:03-cv-02477,KV Pharmaceutical Company | ETHEX Corporation ...,CIMA Labs Inc | Schwarz Pharma Inc | Cima Labs...,6024981
9,739,0:03-cv-02499,"Big Roc Tools, Inc.",Bondhus Corporation,D382190 | 5970828 | 5791211


## 6. litigated_set_int 재구성 — patent_doc_type=='Patent' 필터 + 정규화

In [14]:
def normalize_patent(s):
    if pd.isna(s):
        return None
    s = str(s).strip().upper()
    m = re.match(r'^([A-Z]*)(\d+)$', s)
    if not m:
        return None
    prefix, digits = m.group(1), m.group(2)
    return prefix + digits.lstrip('0')

# case_type_1==1 케이스의 patent_doc_type=='Patent'만
cases_ids = set(cases[cases['case_type_1'] == 1]['case_row_id'])
lit_patents = patents[
    patents['case_row_id'].isin(cases_ids) &
    (patents['patent_doc_type'].str.strip() == 'Patent')
].copy()
lit_patents['patent'] = lit_patents['patent'].astype(str).str.strip()
lit_patents['patent_norm'] = lit_patents['patent'].apply(normalize_patent)

litigated_set      = set(lit_patents['patent'].dropna().unique())
litigated_norm_set = set(lit_patents['patent_norm'].dropna().unique())
litigated_set_int  = {int(p) for p in litigated_set if p.isdigit()}

print(f'소송 특허 unique (원본):  {len(litigated_set):,}건')
print(f'  └ 정수 변환 가능 (PatentsView용): {len(litigated_set_int):,}건')
print(f'  └ prefix 있는 특허 (D/PP/RE 등): {len(litigated_norm_set) - len(litigated_set_int):,}건')

소송 특허 unique (원본):  38,214건
  └ 정수 변환 가능 (PatentsView용): 33,657건
  └ prefix 있는 특허 (D/PP/RE 등): 4,550건


## 7. g_assignee_disambiguated — 소송 특허와 매칭

In [15]:
# 필요한 컬럼만 로드 (875만 행이라 전체 로드는 무거움)
assignee = pd.read_csv(
    f'{DATA_DIR}/g_assignee_disambiguated.tsv',
    sep='\t',
    usecols=['patent_id', 'assignee_sequence', 'disambig_assignee_organization',
             'disambig_assignee_individual_name_first', 'disambig_assignee_individual_name_last',
             'assignee_type'],
    low_memory=False
)
assignee['patent_id'] = assignee['patent_id'].astype(str).str.strip()
print(f'assignee 전체 행 수: {len(assignee):,}')
print(f'unique patent_id 수: {assignee["patent_id"].nunique():,}')
assignee.head(3)

assignee 전체 행 수: 8,751,310
unique patent_id 수: 8,434,880


,patent_id,assignee_sequence,disambig_assignee_individual_name_first,disambig_assignee_individual_name_last,disambig_assignee_organization,assignee_type
0,4488683,0,NaN,NaN,Metal Works Ramat David,3.0
1,11872626,0,NaN,NaN,"DIVERGENT TECHNOLOGIES, INC.",2.0
2,5856666,0,NaN,NaN,U.S. Philips Corporation,2.0


In [11]:
# 소송 특허 목록과 assignee 매칭 (patent_id는 문자열, litigated_set도 문자열)
assignee_in_lit = assignee[assignee['patent_id'].isin(litigated_set)]

matched_patents    = assignee_in_lit['patent_id'].nunique()
unmatched_patents  = len(litigated_set) - matched_patents
match_pct          = matched_patents / len(litigated_set) * 100

print(f'소송 특허 {len(litigated_set):,}건 중')
print(f'  ✔ assignee 있음: {matched_patents:,}건 ({match_pct:.1f}%)')
print(f'  ✘ assignee 없음: {unmatched_patents:,}건 ({100-match_pct:.1f}%)')

소송 특허 38,214건 중
  ✔ assignee 있음: 31,006건 (81.1%)
  ✘ assignee 없음: 7,208건 (18.9%)


In [16]:
# assignee_type 분포 (매칭된 소송 특허만)
# assignee_type: 2=US company, 3=foreign company, 4=US individual, 5=foreign individual, 6=US gov, 7=foreign gov, 9=unassigned
type_map = {
    1: '1-Unassigned',
    2: '2-US Company',
    3: '3-Foreign Company',
    4: '4-US Individual',
    5: '5-Foreign Individual',
    6: '6-US Government',
    7: '7-Foreign Government',
    9: '9-Unassigned'
}
assignee_in_lit['type_label'] = assignee_in_lit['assignee_type'].map(type_map).fillna('기타')
print('=== 소송 특허 assignee_type 분포 (행 기준) ===')
print(assignee_in_lit['type_label'].value_counts())
print()

# assignee가 여러 개인 특허 현황
multi_assignee = assignee_in_lit.groupby('patent_id').size()
print(f'=== 특허당 assignee 수 ===')
print(multi_assignee.value_counts().sort_index().rename('특허 수').to_frame())

NameError: name 'assignee_in_lit' is not defined

In [13]:
# 소송 특허별 assignee 정리 (assignee_sequence==0이 primary)
assignee_primary = assignee_in_lit[assignee_in_lit['assignee_sequence'] == 0].copy()
assignee_primary['assignee_name'] = assignee_primary['disambig_assignee_organization'].fillna(
    assignee_primary['disambig_assignee_individual_name_first'].fillna('') + ' ' +
    assignee_primary['disambig_assignee_individual_name_last'].fillna('')
).str.strip()

assignee_primary = assignee_primary[['patent_id', 'assignee_name', 'assignee_type', 'type_label']].rename(
    columns={'patent_id': 'patent'}
)

print(f'primary assignee 있는 소송 특허: {len(assignee_primary):,}건')
assignee_primary.head(10)

primary assignee 있는 소송 특허: 31,006건


,patent,assignee_name,assignee_type,type_label
169,7534243,MAQUET CARDIOVASCULAR LLC,2.0,2-US Company
429,5439689,Carderm Capital L.P.,3.0,3-Foreign Company
515,8167256,Leslie C. Bishop,4.0,4-US Individual
1120,6632679,Sire Analytical Systems SRL,3.0,3-Foreign Company
2303,6536064,ROM AG,3.0,3-Foreign Company
2622,7289647,"IQ Biometrix, Inc.",2.0,2-US Company
2918,7754127,Let's Gel Incorporated,2.0,2-US Company
2942,7844919,"Versata Development Group, Inc.",2.0,2-US Company
3461,D485880,"NIKE, Inc.",2.0,2-US Company
3527,8490545,DEK Vectorguard Limited,3.0,3-Foreign Company


## 8. 특허 단위 explode → 33,657건 필터 → assignee 매칭 → (p)/(d) 표시

In [14]:
# result 테이블에서 patents 열을 특허 단위로 분리
exploded = (
    result
    .assign(patent=result['patents'].str.split(r'\s*\|\s*'))
    .explode('patent')
    .drop(columns='patents')
    .reset_index(drop=True)
)
exploded['patent'] = exploded['patent'].astype(str).str.strip()

# 정수 변환 가능한 것만 (litigated_set_int 기준)
exploded_int = exploded[exploded['patent'].str.isdigit()].copy()
exploded_int['patent_norm'] = exploded_int['patent'].apply(normalize_patent)

print(f'explode 후 전체 특허 행: {len(exploded):,}')
print(f'정수 변환 가능 (litigated_set_int 기준): {len(exploded_int):,}행  /  unique patent: {exploded_int["patent"].nunique():,}건')
exploded_int.head(5)

explode 후 전체 특허 행: 103,795


정수 변환 가능 (litigated_set_int 기준): 95,520행  /  unique patent: 33,657건


,case_row_id,case_number,defendant,plaintiff,patent,patent_norm
0,727,0:03-cv-00968,Michael Degerness | Advanced Barrel Systems | ...,"Magnum Research, Inc",6457274,6457274
1,727,0:03-cv-00968,Michael Degerness | Advanced Barrel Systems | ...,"Magnum Research, Inc",5600912,5600912
2,728,0:03-cv-00993,Alumacraft Boat Co,"Crestliner, Inc. | Crestliner Inc",5108321,5108321
3,729,0:03-cv-01016,"AS Hospitality, Inc.",Taylor Corporation,6153278,6153278
4,731,0:03-cv-01064,Koninklijke Philips Electronics N.V. | a Nethe...,"Cardiac Science, Inc | a Delaware Corporation ...",5897576,5897576


In [15]:
# assignee_primary 조인 (patent 기준)
df = exploded_int.merge(
    assignee_primary[['patent', 'assignee_name', 'assignee_type', 'type_label']],
    on='patent',
    how='left'
)

# ── 이름 매칭 함수 ─────────────────────────────────────────────────────────────
_STOPWORDS = {
    'inc', 'llc', 'corp', 'ltd', 'co', 'company', 'the', 'and', 'of', 'a', 'an',
    'corporation', 'incorporated', 'limited', 'international', 'group',
    'holdings', 'technologies', 'technology', 'systems', 'solutions',
    'enterprises', 'partners', 'associates', 'industries', 'services'
}

def _tokens(s):
    if pd.isna(s):
        return set()
    s = re.sub(r'[^a-z0-9 ]', ' ', str(s).lower())
    return {t for t in s.split() if len(t) > 2 and t not in _STOPWORDS}

def match_label(assignee, plaintiff, defendant):
    if pd.isna(assignee) or str(assignee).strip() == '':
        return ''
    a_tok = _tokens(assignee)
    if not a_tok:
        return ''

    def hits(party_str):
        if pd.isna(party_str):
            return False
        for party in str(party_str).split('|'):
            p_tok = _tokens(party)
            if len(a_tok & p_tok) >= 2:
                return True
            # 정제 후 substring 포함 여부
            a_s = re.sub(r'[^a-z0-9]', '', str(assignee).lower())
            p_s = re.sub(r'[^a-z0-9]', '', party.lower())
            if a_s and p_s and (a_s in p_s or p_s in a_s):
                return True
        return False

    p_hit = hits(plaintiff)
    d_hit = hits(defendant)
    if p_hit and d_hit:
        return '(p)(d)'
    elif p_hit:
        return '(p)'
    elif d_hit:
        return '(d)'
    return ''

df['match'] = df.apply(
    lambda r: match_label(r['assignee_name'], r['plaintiff'], r['defendant']), axis=1
)
print(f'총 행 수: {len(df):,}')
df.head(5)

총 행 수: 95,520


,case_row_id,case_number,defendant,plaintiff,patent,patent_norm,assignee_name,assignee_type,type_label,match
0,727,0:03-cv-00968,Michael Degerness | Advanced Barrel Systems | ...,"Magnum Research, Inc",6457274,6457274,NaN,NaN,NaN,
1,727,0:03-cv-00968,Michael Degerness | Advanced Barrel Systems | ...,"Magnum Research, Inc",5600912,5600912,NaN,NaN,NaN,
2,728,0:03-cv-00993,Alumacraft Boat Co,"Crestliner, Inc. | Crestliner Inc",5108321,5108321,"Crestliner, Inc.",2.0,2-US Company,(p)
3,729,0:03-cv-01016,"AS Hospitality, Inc.",Taylor Corporation,6153278,6153278,"The Taylor Group, Inc.",2.0,2-US Company,
4,731,0:03-cv-01064,Koninklijke Philips Electronics N.V. | a Nethe...,"Cardiac Science, Inc | a Delaware Corporation ...",5897576,5897576,SurVivaLink Corporation,2.0,2-US Company,


In [16]:
# ── 요약 통계 ─────────────────────────────────────────────────────────────────
no_assignee = df['assignee_name'].isna().sum()
has_assignee = df['assignee_name'].notna().sum()

match_counts = df['match'].value_counts()
matched_p   = match_counts.get('(p)', 0)
matched_d   = match_counts.get('(d)', 0)
matched_pd  = match_counts.get('(p)(d)', 0)
no_match    = match_counts.get('', 0)

print(f'{"전체 특허 행":<30} {len(df):>8,}')
print(f'{"assignee 없음":<30} {no_assignee:>8,}  ({no_assignee/len(df)*100:.1f}%)')
print(f'{"assignee 있음":<30} {has_assignee:>8,}  ({has_assignee/len(df)*100:.1f}%)')
print(f'  ├ Plaintiff 일치  (p)     {matched_p:>8,}  ({matched_p/len(df)*100:.1f}%)')
print(f'  ├ Defendant 일치  (d)     {matched_d:>8,}  ({matched_d/len(df)*100:.1f}%)')
print(f'  ├ 둘 다 일치      (p)(d)  {matched_pd:>8,}  ({matched_pd/len(df)*100:.1f}%)')
print(f'  └ 이름 불일치            {no_match:>8,}  ({no_match/len(df)*100:.1f}%)')

전체 특허 행                          95,520
assignee 없음                      17,257  (18.1%)
assignee 있음                      78,263  (81.9%)
  ├ Plaintiff 일치  (p)       16,695  (17.5%)
  ├ Defendant 일치  (d)        1,375  (1.4%)
  ├ 둘 다 일치      (p)(d)    17,593  (18.4%)
  └ 이름 불일치              59,857  (62.7%)


In [17]:
# 최종 테이블 출력
final = df[['case_row_id', 'case_number', 'defendant', 'plaintiff',
            'patent', 'patent_norm', 'assignee_name', 'match']].copy()
final.head(30)

,case_row_id,case_number,defendant,plaintiff,patent,patent_norm,assignee_name,match
0,727,0:03-cv-00968,Michael Degerness | Advanced Barrel Systems | ...,"Magnum Research, Inc",6457274,6457274,NaN,
1,727,0:03-cv-00968,Michael Degerness | Advanced Barrel Systems | ...,"Magnum Research, Inc",5600912,5600912,NaN,
2,728,0:03-cv-00993,Alumacraft Boat Co,"Crestliner, Inc. | Crestliner Inc",5108321,5108321,"Crestliner, Inc.",(p)
3,729,0:03-cv-01016,"AS Hospitality, Inc.",Taylor Corporation,6153278,6153278,"The Taylor Group, Inc.",
4,731,0:03-cv-01064,Koninklijke Philips Electronics N.V. | a Nethe...,"Cardiac Science, Inc | a Delaware Corporation ...",5897576,5897576,SurVivaLink Corporation,
5,731,0:03-cv-01064,Koninklijke Philips Electronics N.V. | a Nethe...,"Cardiac Science, Inc | a Delaware Corporation ...",5800460,5800460,"Heartstream, Inc.",
6,731,0:03-cv-01064,Koninklijke Philips Electronics N.V. | a Nethe...,"Cardiac Science, Inc | a Delaware Corporation ...",6016059,6016059,"Heartstream, Inc.",
7,731,0:03-cv-01064,Koninklijke Philips Electronics N.V. | a Nethe...,"Cardiac Science, Inc | a Delaware Corporation ...",5591213,5591213,"Heartstream, Inc.",
8,731,0:03-cv-01064,Koninklijke Philips Electronics N.V. | a Nethe...,"Cardiac Science, Inc | a Delaware Corporation ...",5773961,5773961,"Heartstream, Inc.",
9,731,0:03-cv-01064,Koninklijke Philips Electronics N.V. | a Nethe...,"Cardiac Science, Inc | a Delaware Corporation ...",6088616,6088616,SurVivaLink Corporation,


## 9. NPE 후보 탐색 — 이름 불일치 assignee 반복 등장 분석

In [18]:
# match=='' & assignee 있는 행 = 이름 불일치 케이스
no_match_df = df[(df['match'] == '') & df['assignee_name'].notna()].copy()

# assignee별 소송 케이스 수 (unique case_row_id 기준)
npe_candidates = (
    no_match_df.groupby('assignee_name')
    .agg(
        patent_cnt   = ('patent', 'count'),
        case_cnt     = ('case_row_id', 'nunique'),
        type_label   = ('type_label', 'first'),
    )
    .sort_values('case_cnt', ascending=False)
    .reset_index()
)

# 이름 휴리스틱 플래그
_NPE_KEYWORDS = r'holding|licens|patent|acquisit|intellectu|venture|capital|asset|fund|royalt|innovation|technolog.*llc'
npe_candidates['name_flag'] = npe_candidates['assignee_name'].str.lower().str.contains(_NPE_KEYWORDS, regex=True)

print(f'이름 불일치 unique assignee: {len(npe_candidates):,}개')
print(f'이름 휴리스틱 플래그 해당: {npe_candidates["name_flag"].sum():,}개')
print()
print('=== 소송 케이스 수 상위 50 ===')
npe_candidates.head(50)

이름 불일치 unique assignee: 5,600개
이름 휴리스틱 플래그 해당: 264개

=== 소송 케이스 수 상위 50 ===


,assignee_name,patent_cnt,case_cnt,type_label,name_flag
0,"ArrivalStar, Inc.",1669,513,2-US Company,False
1,Xerox Corporation,501,455,2-US Company,False
2,PARADYNE CORPORATION,625,340,2-US Company,False
3,Symantec Corporation,340,259,2-US Company,False
4,AT&T CORP.,261,223,2-US Company,False
5,"Global Research Systems, Inc.",258,179,2-US Company,False
6,"Legalview Assets, Limited",333,157,3-Foreign Company,True
7,Telequip Corporation,156,156,2-US Company,False
8,"Vigilos, LLC",237,152,2-US Company,False
9,"Melvino Technologies, Limited",146,146,3-Foreign Company,False


In [19]:
# 케이스 5건 이상 + 이름 플래그 = 강력 NPE 후보
strong_npe = npe_candidates[(npe_candidates['case_cnt'] >= 5) & npe_candidates['name_flag']]
print(f'강력 NPE 후보 (케이스 5건↑ + 이름 플래그): {len(strong_npe):,}개')
strong_npe

강력 NPE 후보 (케이스 5건↑ + 이름 플래그): 58개


,assignee_name,patent_cnt,case_cnt,type_label,name_flag
6,"Legalview Assets, Limited",333,157,3-Foreign Company,True
23,TRANSACTION HOLDINGS LTD. LLC,921,82,2-US Company,True
27,"Greenwich Information Technologies, LLC",77,77,기타,True
30,International Patent Holdings LLC,75,75,3-Foreign Company,True
33,Klausner Patent Technologies,79,70,2-US Company,True
77,"SRR PATENT HOLDINGS, LLC",62,46,2-US Company,True
90,"WGRS Licensing Company, LLC",39,39,2-US Company,True
104,"DPHI Acquisitions, Inc.",79,34,2-US Company,True
130,Innovation Institute,60,30,2-US Company,True
134,"Accelerate Ventures, L.L.C.",30,30,2-US Company,True


In [20]:
# 전체 df에 NPE 후보 여부 컬럼 추가
npe_set = set(strong_npe['assignee_name'])
npe_set_broad = set(npe_candidates[npe_candidates['case_cnt'] >= 5]['assignee_name'])  # 이름 무관 반복 다수

df['npe_flag']        = df['assignee_name'].isin(npe_set).map({True: 'NPE_후보', False: ''})
df['npe_flag_broad']  = df['assignee_name'].isin(npe_set_broad).map({True: 'NPE_후보(반복)', False: ''})

print('=== NPE 후보 플래그 분포 ===')
print(df['npe_flag'].value_counts())
print()
print('=== 반복 다수(5건↑, 이름 무관) 분포 ===')
print(df['npe_flag_broad'].value_counts())

=== NPE 후보 플래그 분포 ===
npe_flag
          92831
NPE_후보     2689
Name: count, dtype: int64

=== 반복 다수(5건↑, 이름 무관) 분포 ===
npe_flag_broad
              57997
NPE_후보(반복)    37523
Name: count, dtype: int64


## 10. NPE 판별 — 3중 신호 결합 (이름 휴리스틱 + 반복 소송 + Known NPE 리스트)

In [21]:
# ── Signal 3: 학술·언론으로 확인된 Known NPE 리스트 ───────────────────────────
# 출처: RPX "Most Active Patent Trolls", Stanford NPE DB 논문, EFF, 언론 보도
KNOWN_NPE = {
    # 대형 PAE / 특허 펀드
    'Acacia Research', 'Intellectual Ventures', 'IV',
    'RPX Corporation', 'InterDigital',
    'Marathon Patent Group', 'Marathon Technologies',
    'Empire IP', 'Empire IP LLC',
    'Vringo', 'FORM Holdings',
    'Wi-LAN', 'Wi-LAN Inc',
    'Pendrell', 'Clearwire',
    'ParkerVision',
    'Finjan', 'Finjan Holdings',
    'Uniloc', 'Uniloc Luxembourg',
    'IP Edge', 'IP Edge LLC',
    'Rothschild', 'Leigh M. Rothschild',
    'Blue Spike', 'Blue Spike LLC',
    # 유명 개인/소규모 NPE
    'NTP Inc', 'NTP Incorporated',
    'i4i', 'i4i Limited Partnership',
    'Eolas Technologies',
    'Bedrock Computer Technologies',
    'DataTreasury', 'DataTreasury Corporation',
    'Lemelson Medical', 'Lemelson',
    'MercExchange',
    'Forgent Networks',
    'Mirror Worlds', 'Mirror Worlds Technologies',
    'VirnetX', 'VirnetX Inc',
    'Lodsys', 'Lodsys LLC',
    'MPHJ Technology', 'MPHJ Technology Investments',
    'Rovi', 'Rovi Corporation', 'Rovi Guides',
    'TiVo', 'TiVo Inc',
    'Straight Path', 'Straight Path IP',
    'Constellation Technologies',
    'Allacept',
    'Sisvel',
    'MPEG LA',
    # 우리 데이터에서 이미 발견된 것들
    'ArrivalStar', 'ArrivalStar Inc',
    'Legalview Assets', 'Legalview Assets Limited',
    'Transaction Holdings', 'TRANSACTION HOLDINGS LTD. LLC',
    'International Patent Holdings',
    'Greenwich Information Technologies',
    'Melvino Technologies',
    'Vigilos', 'Vigilos LLC',
    'Global Research Systems',
    'REAGAN INVENTIONS',
    'ADVANCED MESSAGING TECHNOLOGIES',
    'VENOVATION',
    'System X',
    'Millennium L.P.',
    'Klausner Patent Technologies',
    'Ariel Inventions',
    'Clear With Computers',
    'Ferrara Ethereal',
    'Empire IP',
    'EpicRealm Licensing',
    'Walker Asset Management',
    'SRR PATENT HOLDINGS',
    'WGRS Licensing',
    'DPHI Acquisitions',
    'Ring Technology Enterprises',
    'Immotion Imagery Technologies',
    'INTELLECTUAL VENTURES FUND',
    'Varis Holdings',
    'Rothschild Trust Holdings',
    'Data Innovation',
    'Graphic Properties Holdings',
    'Samantic Search Technologies',
    'Network Foundation Technologies',
    'Patent Monetization Associates',
    'Brandywine Communications Technologies',
    'Sports Licensing',
    'Timely Innovations',
    'Acceleration Bay',
    'Accelerate Ventures',
}

# 부분 일치용 키워드 (known NPE 이름의 핵심 단어)
_known_lower = {n.lower() for n in KNOWN_NPE}

def is_known_npe(name):
    if pd.isna(name):
        return False
    n = name.lower().strip()
    # 완전 일치
    if n in _known_lower:
        return True
    # 핵심 단어 포함 (긴 것 먼저)
    for k in sorted(_known_lower, key=len, reverse=True):
        if len(k) > 6 and k in n:
            return True
    return False

npe_candidates['known_npe'] = npe_candidates['assignee_name'].apply(is_known_npe)
print(f'known NPE 리스트 매칭: {npe_candidates["known_npe"].sum():,}개')
npe_candidates[npe_candidates['known_npe']].head(30)

known NPE 리스트 매칭: 42개


,assignee_name,patent_cnt,case_cnt,type_label,name_flag,known_npe
0,"ArrivalStar, Inc.",1669,513,2-US Company,False,True
5,"Global Research Systems, Inc.",258,179,2-US Company,False,True
6,"Legalview Assets, Limited",333,157,3-Foreign Company,True,True
8,"Vigilos, LLC",237,152,2-US Company,False,True
9,"Melvino Technologies, Limited",146,146,3-Foreign Company,False,True
14,"REAGAN INVENTIONS, LLC",106,98,2-US Company,False,True
16,"ADVANCED MESSAGING TECHNOLOGIES, INC.",363,94,2-US Company,False,True
23,TRANSACTION HOLDINGS LTD. LLC,921,82,2-US Company,True,True
24,VENOVATION INC.,163,82,2-US Company,False,True
27,"Greenwich Information Technologies, LLC",77,77,기타,True,True


In [22]:
# ── 3중 신호 → NPE 점수 ──────────────────────────────────────────────────────
# signal1: 이름 휴리스틱
# signal2: 반복 소송 (case_cnt >= 5 & match=='')
# signal3: known NPE 리스트

npe_candidates['sig_name']   = npe_candidates['name_flag'].astype(int)
npe_candidates['sig_repeat'] = (npe_candidates['case_cnt'] >= 5).astype(int)
npe_candidates['sig_known']  = npe_candidates['known_npe'].astype(int)
npe_candidates['npe_score']  = (
    npe_candidates['sig_known']  * 3 +   # known은 가장 신뢰도 높음
    npe_candidates['sig_repeat'] * 2 +   # 반복 소송
    npe_candidates['sig_name']   * 1     # 이름 휴리스틱
)

# npe_score 기준 분류
def npe_label(score):
    if score >= 4: return 'NPE_확실'
    if score >= 3: return 'NPE_유력'
    if score >= 2: return 'NPE_의심'
    return '미분류'

npe_candidates['npe_label'] = npe_candidates['npe_score'].apply(npe_label)

print('=== NPE 라벨 분포 ===')
print(npe_candidates['npe_label'].value_counts())
print()
print('=== NPE_확실 상위 목록 ===')
npe_candidates[npe_candidates['npe_label'] == 'NPE_확실'].sort_values('case_cnt', ascending=False).head(30)

=== NPE 라벨 분포 ===


npe_label
미분류       4632
NPE_의심     893
NPE_확실      39
NPE_유력      36
Name: count, dtype: int64

=== NPE_확실 상위 목록 ===


,assignee_name,patent_cnt,case_cnt,type_label,name_flag,known_npe,sig_name,sig_repeat,sig_known,npe_score,npe_label
0,"ArrivalStar, Inc.",1669,513,2-US Company,False,True,0,1,1,5,NPE_확실
5,"Global Research Systems, Inc.",258,179,2-US Company,False,True,0,1,1,5,NPE_확실
6,"Legalview Assets, Limited",333,157,3-Foreign Company,True,True,1,1,1,6,NPE_확실
8,"Vigilos, LLC",237,152,2-US Company,False,True,0,1,1,5,NPE_확실
9,"Melvino Technologies, Limited",146,146,3-Foreign Company,False,True,0,1,1,5,NPE_확실
14,"REAGAN INVENTIONS, LLC",106,98,2-US Company,False,True,0,1,1,5,NPE_확실
16,"ADVANCED MESSAGING TECHNOLOGIES, INC.",363,94,2-US Company,False,True,0,1,1,5,NPE_확실
23,TRANSACTION HOLDINGS LTD. LLC,921,82,2-US Company,True,True,1,1,1,6,NPE_확실
24,VENOVATION INC.,163,82,2-US Company,False,True,0,1,1,5,NPE_확실
27,"Greenwich Information Technologies, LLC",77,77,기타,True,True,1,1,1,6,NPE_확실


In [23]:
# 최종 df에 NPE 라벨 붙이기
npe_label_map = npe_candidates.set_index('assignee_name')['npe_label'].to_dict()
df['npe_label'] = df['assignee_name'].map(npe_label_map).fillna('')

print('=== 전체 특허 행 기준 NPE 라벨 분포 ===')
print(df['npe_label'].value_counts())
print()

# 최종 테이블 (NPE_확실만 샘플)
cols = ['case_row_id', 'case_number', 'plaintiff', 'defendant',
        'patent', 'assignee_name', 'match', 'npe_label']
df[df['npe_label'] == 'NPE_확실'][cols].sort_values('case_row_id').head(30)

=== 전체 특허 행 기준 NPE 라벨 분포 ===
npe_label
          39121
NPE_의심    30737
미분류       18392
NPE_확실     6194
NPE_유력     1076
Name: count, dtype: int64



,case_row_id,case_number,plaintiff,defendant,patent,assignee_name,match,npe_label
190,845,0:04-cv-00907,"Arrival Star, Inc. | a foreign corporation",Nistevo Corporation | a domestic corporationTE...,5400020,"Global Research Systems, Inc.",,NPE_확실
191,845,0:04-cv-00907,"Arrival Star, Inc. | a foreign corporation",Nistevo Corporation | a domestic corporationTE...,6486801,"ArrivalStar, Inc.",(p)(d),NPE_확실
192,845,0:04-cv-00907,"Arrival Star, Inc. | a foreign corporation",Nistevo Corporation | a domestic corporationTE...,6683542,"ArrivalStar, Inc.",(p)(d),NPE_확실
193,845,0:04-cv-00907,"Arrival Star, Inc. | a foreign corporation",Nistevo Corporation | a domestic corporationTE...,5657010,"Global Research Systems, Inc.",,NPE_확실
194,845,0:04-cv-00907,"Arrival Star, Inc. | a foreign corporation",Nistevo Corporation | a domestic corporationTE...,5623260,"Global Research Systems, Inc.",,NPE_확실
195,845,0:04-cv-00907,"Arrival Star, Inc. | a foreign corporation",Nistevo Corporation | a domestic corporationTE...,6317060,"Global Research Systems, Inc.",,NPE_확실
196,845,0:04-cv-00907,"Arrival Star, Inc. | a foreign corporation",Nistevo Corporation | a domestic corporationTE...,6363323,"Global Research Systems, Inc.",,NPE_확실
197,845,0:04-cv-00907,"Arrival Star, Inc. | a foreign corporation",Nistevo Corporation | a domestic corporationTE...,6411891,"Global Research Systems, Inc.",,NPE_확실
198,845,0:04-cv-00907,"Arrival Star, Inc. | a foreign corporation",Nistevo Corporation | a domestic corporationTE...,6415207,"Global Research Systems, Inc.",,NPE_확실
199,845,0:04-cv-00907,"Arrival Star, Inc. | a foreign corporation",Nistevo Corporation | a domestic corporationTE...,6510383,"ArrivalStar, Inc.",(p)(d),NPE_확실


## 11. 최종 레이블 테이블 저장

In [24]:
final_labeled = df[[
    'case_row_id', 'case_number',
    'plaintiff', 'defendant',
    'patent', 'patent_norm',
    'assignee_name', 'type_label',
    'match',        # (p) / (d) / (p)(d) / ''
    'npe_label',    # NPE_확실 / NPE_유력 / NPE_의심 / 미분류 / ''
]].copy()

# match 빈 문자열 → 명시적 레이블
final_labeled['match'] = final_labeled['match'].replace('', 'unmatched')
final_labeled['npe_label'] = final_labeled['npe_label'].replace('', 'non-NPE')

print('=== 최종 레이블 분포 ===')
print(final_labeled['match'].value_counts())
print()
print(final_labeled['npe_label'].value_counts())
print()
print(f'총 행 수: {len(final_labeled):,}')
final_labeled.head(20)

=== 최종 레이블 분포 ===
match
unmatched    59857
(p)(d)       17593
(p)          16695
(d)           1375
Name: count, dtype: int64

npe_label
non-NPE    39121
NPE_의심     30737
미분류        18392
NPE_확실      6194
NPE_유력      1076
Name: count, dtype: int64

총 행 수: 95,520


,case_row_id,case_number,plaintiff,defendant,patent,patent_norm,assignee_name,type_label,match,npe_label
0,727,0:03-cv-00968,"Magnum Research, Inc",Michael Degerness | Advanced Barrel Systems | ...,6457274,6457274,NaN,NaN,unmatched,non-NPE
1,727,0:03-cv-00968,"Magnum Research, Inc",Michael Degerness | Advanced Barrel Systems | ...,5600912,5600912,NaN,NaN,unmatched,non-NPE
2,728,0:03-cv-00993,"Crestliner, Inc. | Crestliner Inc",Alumacraft Boat Co,5108321,5108321,"Crestliner, Inc.",2-US Company,(p),non-NPE
3,729,0:03-cv-01016,Taylor Corporation,"AS Hospitality, Inc.",6153278,6153278,"The Taylor Group, Inc.",2-US Company,unmatched,미분류
4,731,0:03-cv-01064,"Cardiac Science, Inc | a Delaware Corporation ...",Koninklijke Philips Electronics N.V. | a Nethe...,5897576,5897576,SurVivaLink Corporation,2-US Company,unmatched,미분류
5,731,0:03-cv-01064,"Cardiac Science, Inc | a Delaware Corporation ...",Koninklijke Philips Electronics N.V. | a Nethe...,5800460,5800460,"Heartstream, Inc.",2-US Company,unmatched,NPE_의심
6,731,0:03-cv-01064,"Cardiac Science, Inc | a Delaware Corporation ...",Koninklijke Philips Electronics N.V. | a Nethe...,6016059,6016059,"Heartstream, Inc.",2-US Company,unmatched,NPE_의심
7,731,0:03-cv-01064,"Cardiac Science, Inc | a Delaware Corporation ...",Koninklijke Philips Electronics N.V. | a Nethe...,5591213,5591213,"Heartstream, Inc.",2-US Company,unmatched,NPE_의심
8,731,0:03-cv-01064,"Cardiac Science, Inc | a Delaware Corporation ...",Koninklijke Philips Electronics N.V. | a Nethe...,5773961,5773961,"Heartstream, Inc.",2-US Company,unmatched,NPE_의심
9,731,0:03-cv-01064,"Cardiac Science, Inc | a Delaware Corporation ...",Koninklijke Philips Electronics N.V. | a Nethe...,6088616,6088616,SurVivaLink Corporation,2-US Company,unmatched,미분류


In [25]:
final_labeled.to_csv('data/patent_cases_labeled.csv', index=False)
print('저장 완료: data/patent_cases_labeled.csv')
print(f'shape: {final_labeled.shape}')

저장 완료: data/patent_cases_labeled.csv
shape: (95520, 10)


## 12. lead_case 탐색 — 병합 소송 구조 파악

In [26]:
# lead_case: PACER가 여러 케이스를 하나의 소송으로 공식 병합할 때 기재
# 같은 lead_case를 공유하는 case_row_id들 = 동일 분쟁의 복수 제소
cases_full = pd.read_csv(f'{DATA_DIR}/cases.csv',
    usecols=['case_row_id','case_number','case_type_1','date_filed','lead_case','related_case'],
    low_memory=False)
cases_t1 = cases_full[cases_full['case_type_1']==1].copy()
cases_t1['date_filed'] = pd.to_datetime(cases_t1['date_filed'], errors='coerce')

print(f'case_type_1==1 전체: {len(cases_t1):,}건')
print(f'lead_case 있음:      {cases_t1["lead_case"].notna().sum():,}건')
print(f'related_case 있음:   {cases_t1["related_case"].notna().sum():,}건')
print()
print('=== lead_case 샘플 ===')
cases_t1[cases_t1['lead_case'].notna()][
    ['case_row_id','case_number','date_filed','lead_case']].head(15)

case_type_1==1 전체: 46,907건
lead_case 있음:      3,330건
related_case 있음:   12,602건

=== lead_case 샘플 ===


,case_row_id,case_number,date_filed,lead_case
454,1182,0:07-cv-04551,2007-11-07,0:07-cv-01687-JNE-JJG
531,1259,0:08-cv-06292,2008-12-08,0:08-cv-06291-PJS-RLE
550,1278,0:09-cv-00545,2009-03-06,0:09-cv-00122-DWF-JSM
629,1357,0:10-cv-01571,2010-04-15,0:10-cv-00687-SRN-JJK
712,1441,0:11-cv-00675,2011-03-18,0:11-md-02249-DWF-SER
748,1477,0:11-cv-02549,2011-09-07,0:11-md-02249-DWF-SER
749,1478,0:11-cv-02550,2011-09-07,0:11-md-02249-DWF-SER
750,1479,0:11-cv-02551,2011-09-09,0:11-md-02249-DWF-SER
751,1480,0:11-cv-02552,2011-09-09,0:11-md-02249-DWF-SER
752,1481,0:11-cv-02553,2011-09-07,0:11-md-02249-DWF-SER


In [27]:
# lead_case 값 = 다른 case_number를 참조
# 같은 lead_case를 가진 케이스들이 하나의 소송 그룹
lead_groups = (cases_t1[cases_t1['lead_case'].notna()]
    .groupby('lead_case')
    .agg(
        n_cases      = ('case_row_id', 'count'),
        case_ids     = ('case_row_id', list),
        case_numbers = ('case_number', list),
        earliest     = ('date_filed', 'min'),
        latest       = ('date_filed', 'max'),
    )
    .reset_index()
    .sort_values('n_cases', ascending=False))

print(f'lead_case 고유 값 수: {len(lead_groups):,}')
print(f'그룹당 케이스 수 분포:')
print(lead_groups['n_cases'].value_counts().sort_index())
print()
lead_groups.head(20)

lead_case 고유 값 수: 1,019
그룹당 케이스 수 분포:
n_cases
1     588
2     153
3      73
4      36
5      26
6      14
7      18
8      24
9       9
10     11
11      8
12      7
13      6
14      6
15      5
16      3
17      2
18      6
19      2
20      1
21      2
22      1
23      1
24      1
25      2
27      2
28      3
31      1
32      1
33      1
37      1
42      1
48      1
49      1
55      1
67      1
Name: count, dtype: int64



,lead_case,n_cases,case_ids,case_numbers,earliest,latest
789,3:13-cv-01944-MLC-DEA,67,"[57388, 57389, 57390, 57391, 57392, 57393, 573...","[3:13-cv-01949, 3:13-cv-01950, 3:13-cv-01951, ...",2013-03-27,2014-08-07
788,3:13-cv-00651-JLS-MDD,55,"[57165, 57166, 57203, 57224, 57225, 57228, 572...","[3:13-cv-00652, 3:13-cv-00653, 3:13-cv-00825, ...",2013-03-19,2013-10-10
446,2:07-ml-01816-RGK -FFM; 2:07-ml-01816-RGK-FFM,49,"[36801, 36802, 36804, 36810, 36817, 36818, 368...","[2:07-cv-02096, 2:07-cv-02099, 2:07-cv-02101, ...",2007-03-30,2007-10-29
240,1:12-cv-06286,48,"[20230, 20233, 20236, 20242, 20781, 20782, 207...","[1:12-cv-01488, 1:12-cv-01489, 1:12-cv-01490, ...",2012-03-01,2013-08-06
600,2:15-cv-00950-JRG,42,"[46398, 46401, 46402, 46403, 46404, 46405, 464...","[2:15-cv-00949, 2:15-cv-00951, 2:15-cv-00952, ...",2015-06-05,2015-06-19
771,3:11-cv-01810-DMS-WVG,37,"[56504, 56505, 56506, 56507, 56508, 56510, 565...","[3:12-cv-00729, 3:12-cv-00731, 3:12-cv-00732, ...",2012-03-27,2014-05-27
896,6:11-cv-00487-LED,33,"[69006, 69007, 69008, 69009, 69010, 69011, 690...","[6:12-cv-00702, 6:12-cv-00703, 6:12-cv-00704, ...",2012-09-27,2012-09-27
952,6:15-cv-00618-JRG-JDL,32,"[70555, 70557, 70558, 70560, 70561, 70564, 705...","[6:15-cv-00616, 6:15-cv-00619, 6:15-cv-00620, ...",2015-07-01,2015-07-02
925,6:13-cv-00626-RWS,31,"[69572, 69573, 69574, 69668, 69669, 69670, 696...","[6:13-cv-00627, 6:13-cv-00628, 6:13-cv-00629, ...",2013-09-06,2014-06-23
575,2:15-cv-00071-RWS-RSP,28,"[45421, 45426, 45428, 45432, 45470, 45471, 454...","[2:15-cv-00073, 2:15-cv-00076, 2:15-cv-00078, ...",2015-01-26,2015-07-27


In [28]:
# 그룹 내 케이스들의 특허가 같은지 확인
patents_raw = pd.read_csv(f'{DATA_DIR}/patents.csv',
    usecols=['case_row_id','patent','patent_doc_type'], low_memory=False)
patents_raw = patents_raw[patents_raw['patent_doc_type'].str.strip()=='Patent'].copy()
patents_raw['patent'] = patents_raw['patent'].astype(str).str.strip()
case2pats = patents_raw.groupby('case_row_id')['patent'].apply(set).to_dict()

results = []
for _, row in lead_groups[lead_groups['n_cases'] >= 2].head(30).iterrows():
    pat_sets = [case2pats.get(cid, set()) for cid in row['case_ids']]
    pat_sets = [s for s in pat_sets if s]
    if len(pat_sets) < 2:
        results.append({'lead_case': row['lead_case'], 'n_cases': row['n_cases'],
                        'all_same': None, 'overlap_rate': None, 'note': '특허 없음'})
        continue
    union = set.union(*pat_sets)
    inter = set.intersection(*pat_sets)
    overlap = len(inter) / len(union) if union else 0
    results.append({
        'lead_case'   : row['lead_case'],
        'n_cases'     : row['n_cases'],
        'case_numbers': row['case_numbers'],
        'all_same'    : overlap == 1.0,
        'overlap_rate': round(overlap, 3),
        'total_unique_patents': len(union),
        'shared_patents': len(inter),
    })

res = pd.DataFrame(results)
print('=== lead_case 그룹 내 특허 일치율 ===')
print(res['overlap_rate'].describe().round(3))
print()
print(f'그룹 내 특허 완전 동일: {res["all_same"].sum():.0f}건')
print(f'그룹 내 특허 일부 겹침: {((res["overlap_rate"]>0)&(res["overlap_rate"]<1)).sum():.0f}건')
print(f'그룹 내 특허 완전 다름: {(res["overlap_rate"]==0).sum():.0f}건')
print()
res[['lead_case','n_cases','case_numbers','overlap_rate','total_unique_patents','shared_patents']].head(30)

=== lead_case 그룹 내 특허 일치율 ===
count    30.000
mean      0.625
std       0.430
min       0.000
25%       0.213
50%       0.862
75%       1.000
max       1.000
Name: overlap_rate, dtype: float64

그룹 내 특허 완전 동일: 14건
그룹 내 특허 일부 겹침: 9건
그룹 내 특허 완전 다름: 7건



,lead_case,n_cases,case_numbers,overlap_rate,total_unique_patents,shared_patents
0,3:13-cv-01944-MLC-DEA,67,"[3:13-cv-01949, 3:13-cv-01950, 3:13-cv-01951, ...",0.000,14,0
1,3:13-cv-00651-JLS-MDD,55,"[3:13-cv-00652, 3:13-cv-00653, 3:13-cv-00825, ...",0.000,4,0
2,2:07-ml-01816-RGK -FFM; 2:07-ml-01816-RGK-FFM,49,"[2:07-cv-02096, 2:07-cv-02099, 2:07-cv-02101, ...",0.000,52,0
3,1:12-cv-06286,48,"[1:12-cv-01488, 1:12-cv-01489, 1:12-cv-01490, ...",0.000,6,0
4,2:15-cv-00950-JRG,42,"[2:15-cv-00949, 2:15-cv-00951, 2:15-cv-00952, ...",1.000,1,1
5,3:11-cv-01810-DMS-WVG,37,"[3:12-cv-00729, 3:12-cv-00731, 3:12-cv-00732, ...",0.000,6,0
6,6:11-cv-00487-LED,33,"[6:12-cv-00702, 6:12-cv-00703, 6:12-cv-00704, ...",1.000,1,1
7,6:15-cv-00618-JRG-JDL,32,"[6:15-cv-00616, 6:15-cv-00619, 6:15-cv-00620, ...",0.750,4,3
8,6:13-cv-00626-RWS,31,"[6:13-cv-00627, 6:13-cv-00628, 6:13-cv-00629, ...",1.000,1,1
9,2:15-cv-00071-RWS-RSP,28,"[2:15-cv-00073, 2:15-cv-00076, 2:15-cv-00078, ...",1.000,1,1
